# 第 7 章 其他监督学习算法（了解）

本章主要是“认识型”内容：**朴素贝叶斯、决策树、支持向量机和集成学习**。你可以把它们理解成：
“除了线性模型之外，还常见的几类监督学习算法家族”。

## 7.1 朴素贝叶斯（Naive Bayes）

### 7.1.1 核心概念

朴素贝叶斯是一类 **基于概率** 的分类算法，核心两点：

1. 用 **贝叶斯定理** 计算“给定特征属于某个类别的概率”；
2. 假设特征之间相互独立（这就是“朴素”）。

**贝叶斯定理：**

$$
P(Y \mid X) = \frac{P(X \mid Y),P(Y)}{P(X)}
$$

* $P(Y\mid X)$：**后验概率**，在看到特征 $X$ 之后，样本属于类别 $Y$ 的概率；
* $P(X\mid Y)$：**条件概率**，在类别为 $Y$ 的样本中，出现特征 $X$ 的概率；
* $P(Y)$：**先验概率**，不看特征，类别 $Y$ 出现的比例；
* $P(X)$：特征 $X$ 出现的总体概率（做分类时通常当作归一化常数）。

在垃圾邮件识别中，如果

* $Y=\text{“垃圾邮件”}$，
* $X=\text{邮件中出现“免费”}$，

那么：

* $P(Y)$：邮件本身是垃圾邮件的概率；
* $P(X\mid Y)$：垃圾邮件中出现“免费”的概率；
* $P(Y\mid X)$：出现“免费”的邮件是垃圾邮件的概率（我们真正关心的）。

**朴素假设（特征独立）：**

若样本有多个特征 $X_1,\dots,X_n$，朴素贝叶斯假设它们在给定类别 $Y$ 时相互独立：

$$
P(X_1,\dots,X_n \mid Y) = \prod_{j=1}^n P(X_j \mid Y)
$$

这样就不用估计一个高维联合概率，只要估计每个特征的条件概率，大大简化计算。

**分类规则：**

对每个类别 $C_k$ 计算：

$$
P(Y = C_k \mid X = x) \propto P(Y = C_k) \prod_{j=1}^n P(X_j = x_j \mid Y = C_k)
$$

选概率最大的那个类别：

$$
\hat y = \arg\max_k P(Y = C_k) \prod_{j=1}^n P(X_j = x_j \mid Y = C_k)
$$

### 7.1.2 极大似然估计 & 贝叶斯估计（平滑）

在朴素贝叶斯中，需要估计两类概率：

* **先验概率**：$P(Y=C_k)$
* **条件概率**：$P(X_j = a_{jl} \mid Y = C_k)$

#### 1）极大似然估计（MLE）

简单理解：**出现几次就除以总次数**。

设训练集中有 $N$ 个样本，类别总数为 $K$。
记 $I(\cdot)$ 为指示函数（条件成立为 1，否则为 0）。

**先验概率的 MLE：**

$$
P(Y = C_k) = \frac{\sum_{i=1}^N I(y_i = C_k)}{N}, \quad k = 1,2,\dots,K
$$

**条件概率的 MLE：**

设第 $j$ 个特征 $X_j$ 的某个取值为 $a_{jl}$，则

$$
P(X_j = a_{jl} \mid Y = C_k)
= \frac{\sum_{i=1}^N I(x_{ji} = a_{jl},, y_i = C_k)}{\sum_{i=1}^N I(y_i = C_k)}
$$

问题：如果某个组合在训练集中 **从未出现**，对应的概率就是 0，
会让整个乘积变成 0，导致分类严重偏差。

#### 2）贝叶斯估计（加平滑）

常见做法是加一个平滑项 $\lambda \ge 0$（常用 $\lambda = 1$，即拉普拉斯平滑）。

**先验概率的贝叶斯估计：**

$$
P_\lambda(Y = C_k) =
\frac{\sum_{i=1}^N I(y_i = C_k) + \lambda}{N + K\lambda}
$$

**条件概率的贝叶斯估计：**

假设第 $j$ 个特征有 $L$ 种可能取值：

$$
P_\lambda(X_j = a_{jl} \mid Y = C_k)
= \frac{\sum_{i=1}^N I(x_{ji} = a_{jl},, y_i = C_k) + \lambda}
{\sum_{i=1}^N I(y_i = C_k) + L\lambda}
$$

* $\lambda = 0$：退化为极大似然估计；
* $\lambda = 1$：拉普拉斯平滑（“每种情况先假装出现过 1 次”）。

### 7.1.3 朴素贝叶斯的学习与分类步骤

**学习阶段（训练）：**

1. 统计每个类别在训练集中的频率，得到先验概率 $P(Y=C_k)$；
2. 对每个特征、每个取值、每个类别计数，得到条件概率 $P(X_j=a_{jl} \mid Y=C_k)$；
3. 视情况对上面的概率做平滑（如拉普拉斯平滑）。

**预测阶段（分类）：**

给定一个新样本 $x = (x_1, \dots, x_n)$：

1. 对每个类别 $C_k$，计算
   $$
   P(Y = C_k)\prod_{j=1}^n P(X_j = x_j \mid Y = C_k)
   $$
2. 选取值最大的类别作为预测结果。

### 7.1.4 朴素贝叶斯代码示例（文本分类）

下面用 `scikit-learn` 做一个简单的垃圾邮件示例（伪数据）：

In [1]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# 1. 准备数据：0 表示正常邮件，1 表示垃圾邮件
texts = [
    "免费 中奖 点击 领取",
    "本周 项目 讨论 会议",
    "优惠 券 免费 领 现在 下单",
    "请 明天 带 上 报告",
]
labels = [1, 0, 1, 0]

# 2. 文本向量化：统计每个词出现次数
vec = CountVectorizer()
X = vec.fit_transform(texts)

# 3. 训练朴素贝叶斯分类器
clf = MultinomialNB()
clf.fit(X, labels)

# 4. 预测新邮件
new_mail = ["免费 优惠 点击 领取 奖品"]
X_new = vec.transform(new_mail)
pred = clf.predict(X_new)

print("预测类别：", pred[0])  # 1 代表垃圾邮件

预测类别： 1


* `CountVectorizer` 把文本转成“词频向量”；
* `MultinomialNB` 是多项式朴素贝叶斯，适合词频这种计数型特征；
* 训练好模型后，对新邮件进行同样的向量化，再用 `predict` 得到类别。

**应用场景：**
朴素贝叶斯适合 **特征很多、样本较多、对速度要求高** 的场景，比如文本分类、垃圾邮件识别、情感分析等。

## 7.2 决策树（Decision Tree）

### 7.2.1 决策树简介

决策树是一种“**像人类写规则一样**”的模型，用一棵树来做判断：

* **根节点**：包含所有样本；
* **内部节点**：基于某个特征做判断（例如“年龄 > 30？”）；
* **分支**：根据判断结果走不同的路径；
* **叶节点**：最终的分类结果（或回归值）。

优点：

* 结果是清晰的规则，**可解释性强**；
* 支持离散特征和连续特征；
* 能处理分类和回归任务。

常见的决策树算法：

* ID3：用**信息增益**选特征；
* C4.5：用**信息增益率**选特征；
* CART：用**基尼指数**（分类）或**平方误差**（回归）。

### 7.2.2 决策树的工作流程

大致过程：

1. 从根节点开始，把所有训练样本放在一起；
2. 在所有特征中挑一个“最能区分数据”的特征（用信息增益、基尼等指标）；
3. 按这个特征的不同取值，把数据分成几个子集，生成子节点；
4. 对每个子节点重复上述步骤：继续选特征、继续划分；
5. 当某个节点的数据已经“比较纯”（大部分都属同一类）或没有特征可用时，停止划分，把该节点设成叶节点；
6. 得到一棵完整的树。

**过拟合问题：**

* 树长得太深、叶子太多，就可能 “记住了训练集”，在新数据上表现不好；
* 解决办法：**剪枝**，让树变小、规则更简单。

### 7.2.3 特征选择的几个指标

#### 1）信息熵

信息熵度量不确定性。设离散随机变量 $X$ 的分布为：

$$
P(X = x_i) = p_i,\quad i = 1,2,\dots,n
$$

则熵为：

$$
H(X) = -\sum_{i=1}^n p_i \log_2 p_i
$$

* 熵越大，表示越“乱”，越不确定；
* 熵为 0 表示完全确定（全是同一类）。

条件熵 $H(Y\mid X)$ 表示在知道 $X$ 的情况下，$Y$ 还有多少不确定性。

#### 2）信息增益（ID3）

给定数据集 $D$ 和特征 $A$：

* $H(D)$：按类别划分时的数据熵；
* $H(D\mid A)$：在已知特征 $A$ 的值后，数据的条件熵；

**信息增益：**

$$
g(D, A) = H(D) - H(D \mid A)
$$

含义：使用特征 $A$ 划分数据后，**不确定性减少了多少**。
ID3 在每个节点选择信息增益最大的特征。

#### 3）信息增益率（C4.5）

信息增益偏向于选择取值很多的特征（比如 ID 号），
C4.5 使用 **信息增益率**：

$$
g_R(D, A) = \frac{g(D, A)}{H_A(D)}
$$

其中 $H_A(D)$ 是“特征 $A$ 的取值熵”，用于惩罚取值过多的特征。

#### 4）基尼指数（Gini）与 CART

对分类问题，样本属于第 $k$ 类的概率为 $p_k$，则：

$$
\text{Gini} = 1 - \sum_{k=1}^K p_k^2
$$

对给定样本集合 $D$，若 $|C_k|$ 是第 $k$ 类的样本数，$|D|$ 为样本总数：

$$
\text{Gini}(D) = 1 - \sum_{k=1}^K \left(\frac{|C_k|}{|D|}\right)^2
$$

特征 $A$ 根据取值把 $D$ 划分为 $D_1, D_2$，其加权基尼指数：

$$
\text{Gini}(D, A) = \frac{|D_1|}{|D|}\text{Gini}(D_1) + \frac{|D_2|}{|D|}\text{Gini}(D_2)
$$

CART 分类树选择 **使 $\text{Gini}(D, A)$ 最小** 的特征与划分点。

#### 5）CART 回归树（最小二乘回归树）

回归树要预测的是连续值，CART 的做法是：

1. 选一个特征 $j$ 和一个切分点 $s$，把输入空间划分为两块：
   $$
   R_1(j,s) = {x \mid x_j \le s},\quad
   R_2(j,s) = {x \mid x_j > s}
   $$

2. 在每个区域上预测一个常数 $c_1, c_2$，用平方误差作为损失：

   $$
   \min_{j,s}
   \left(
   \min_{c_1} \sum_{x_i\in R_1(j,s)} (y_i - c_1)^2
   +
   \min_{c_2} \sum_{x_i\in R_2(j,s)} (y_i - c_2)^2
   \right)
   $$

3. 得到最优 $(j,s)$ 后，继续对每个子区域递归划分，直到满足停止条件；

4. 最终叶节点的预测值通常是该区域样本目标值的均值。

### 7.2.4 剪枝（防止过拟合）

**为什么剪枝？**

* 决策树如果无限长下去，很容易对训练数据“记太熟”，泛化性能下降；
* 剪枝就是“砍掉不必要的枝条”，让模型更简单、更稳。

**两种剪枝方式：**

1. **预剪枝（Pre-Pruning）**：在长树的过程中就限制生长：

   * 限制最大深度 `max_depth`;
   * 限制每个节点最小样本数 `min_samples_leaf`;
   * 限制最小信息增益/误差降低；
   * 限制最大叶子数等。
2. **后剪枝（Post-Pruning）**：先生成一棵“完全树”，
   再自底向上尝试把子树替换为叶节点，如果在验证集上的表现没有变差甚至变好，就剪掉子树。
   常见方法：代价复杂度剪枝（CCP）、减少误差剪枝（REP）等。

### 7.2.5 决策树代码示例（分类）

In [2]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.datasets import load_iris

# 1. 加载示例数据集：鸢尾花数据
X, y = load_iris(return_X_y=True)

# 2. 训练决策树
clf = DecisionTreeClassifier(
    criterion="gini",   # 或 "entropy"
    max_depth=3,        # 预剪枝：限制最大深度
    random_state=0
)
clf.fit(X, y)

# 3. 查看规则（文本形式）
feature_names = load_iris().feature_names
tree_rules = export_text(clf, feature_names=feature_names)
print(tree_rules)

|--- petal width (cm) <= 0.80
|   |--- class: 0
|--- petal width (cm) >  0.80
|   |--- petal width (cm) <= 1.75
|   |   |--- petal length (cm) <= 4.95
|   |   |   |--- class: 1
|   |   |--- petal length (cm) >  4.95
|   |   |   |--- class: 2
|   |--- petal width (cm) >  1.75
|   |   |--- petal length (cm) <= 4.85
|   |   |   |--- class: 2
|   |   |--- petal length (cm) >  4.85
|   |   |   |--- class: 2



**应用场景：**
决策树适合需要 **规则清晰、可解释性高** 的场景，也常作为集成方法（随机森林、梯度提升树）的基础模型。

## 7.3 支持向量机（SVM）

### 7.3.1 支持向量机简介

支持向量机是一个 **二分类模型**，核心思想：

> 找一条（超）平面，把两类样本分开，并且让“离平面最近的点”距离尽可能大（间隔最大）。

* 在 2D 中：超平面是直线；
* 在 3D 中：超平面是平面；
* 在更高维中：超平面是超平面。

间隔大，意味着对噪声更不敏感、**泛化能力更强**。
SVM 模型从简单到复杂可以分为：

1. 线性可分 SVM（硬间隔）；
2. 线性 SVM（软间隔）；
3. 非线性 SVM（核函数）。

### 7.3.2 线性可分支持向量机（硬间隔）

在样本空间中，超平面可以写为：

$$
w^\top x + b = 0
$$

* $w$ 是法向量（决定方向）；
* $b$ 是偏置（决定位置）。

对应分类函数：

$$
f(x) = \operatorname{sign}(w^\top x + b)
$$

设 $x'$ 是平面上的任意一点，$w^\top x' + b = 0$，
任意点 $x$ 到超平面的**几何间隔**为：

$$
r = \frac{|w^\top x + b|}{|w|}
$$

对样本 $(x_i, y_i)$，$y_i \in {+1,-1}$，函数间隔为：

$$
\gamma_i = y_i (w^\top x_i + b)
$$

如果分类正确且有一定余量，则：

$$
y_i(w^\top x_i + b) \ge 1
$$

此时几何间隔为：

$$
r_i = \frac{\gamma_i}{|w|}
$$

通过缩放 $(w,b)$ 不会改变分类结果，但会改变 $\gamma_i$，
因此可以约束支持向量的函数间隔为 1，使得间隔为：

$$
\gamma = \frac{2}{|w|}
$$

于是 **最大化间隔**等价于 **最小化 $\frac{1}{2}|w|^2$**：

$$
\begin{align}
\min_{w,b} \quad & \frac{1}{2}|w|^2 \
\text{s.t.} \quad & y_i(w^\top x_i + b) \ge 1,\quad i=1,\dots,N
\end{align}
$$

这就是线性可分 SVM 的基本形式（硬间隔 SVM）。

### 7.3.3 线性支持向量机（软间隔）

现实中数据往往 **线性不可分**，
允许少量样本分类错误更合理，这就是“软间隔”。

引入松弛变量 $\xi_i \ge 0$，放宽约束：

$$
y_i(w^\top x_i + b) \ge 1 - \xi_i
$$

并在目标中对 $\xi_i$ 做惩罚：

$$
\begin{align}
\min_{w,b,\xi} \quad & \frac{1}{2}|w|^2 + C \sum_{i=1}^n \xi_i \
\text{s.t.} \quad & y_i(w^\top x_i + b) \ge 1 - \xi_i, \
& \xi_i \ge 0,\quad i=1,\dots,n
\end{align}
$$

其中 $C > 0$ 为惩罚系数：

* $C$ 大：更不容忍误差（间隔可能变小）；
* $C$ 小：允许更多误差（间隔可以更大）。

### 7.3.4 非线性支持向量机与核函数

有些数据在原空间中**怎么画直线都分不开**，
但如果把数据映射到一个更高维的空间中，可能就线性可分了。

* 显式做高维映射代价太高；
* SVM 使用 **核技巧（kernel trick）**：
  不显式算特征，只在算法中用核函数 $\kappa(x_i, x_j)$，
  它等价于计算高维空间中映射后的内积。

常见核函数：

* 线性核：
  $$
  \kappa(x_i, x_j) = x_i^\top x_j
  $$
* 多项式核：
  $$
  \kappa(x_i, x_j) = (x_i^\top x_j)^d
  $$
* 高斯核（RBF）：
  $$
  \kappa(x_i, x_j) = \exp\left(-\frac{|x_i - x_j|^2}{2\sigma^2}\right)
  $$
* 拉普拉斯核：
  $$
  \kappa(x_i, x_j) = \exp\left(-\frac{|x_i - x_j|}{\sigma}\right)
  $$
* Sigmoid 核：
  $$
  \kappa(x_i, x_j) = \tanh(\beta, x_i^\top x_j + \theta)
  $$

不同核函数相当于选择了不同的“高维特征空间”。

### 7.3.5 SVM 代码示例（RBF 核）

In [3]:
from sklearn import svm
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. 造一个二分类数据集
X, y = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=0
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

# 2. 使用 RBF 核的 SVM
clf = svm.SVC(kernel="rbf", C=1.0, gamma="scale")
clf.fit(X_train, y_train)

# 3. 评估
y_pred = clf.predict(X_test)
print("测试集准确率：", accuracy_score(y_test, y_pred))

测试集准确率： 0.9


**应用场景：**
SVM 适合特征维度较高、样本规模中等且对边界要求严格的分类任务，
比如文本分类、小规模图像分类等。

## 7.4 集成学习（Ensemble Learning）

### 7.4.1 基本概念

集成学习的核心想法：

> 多个“弱一点”的模型，搭配合适的组合策略，可以得到一个“很强”的模型。

* **同质集成**：个体学习器类型相同（如全是决策树），
  个体称为 **基学习器（base learner）**。
* **异质集成**：个体学习器类型不同（如决策树 + SVM + 神经网络）。

经典三种方式：

1. **Boosting**：顺序训练模型，每一轮更关注前面没学好的样本；
2. **Bagging**：对样本做有放回抽样，训练许多独立模型，最后投票或平均；
3. **Stacking**：多种模型并行训练，然后再训练一个“元模型”来综合它们的输出。

### 7.4.2 AdaBoost（Boosting 代表）

**弱学习器 vs 强学习器：**

* 弱学习器：比随机猜稍微好一点的模型（比如单层决策树桩）；
* 强学习器：能达到较高精度的模型。

Boosting 就是：**用很多弱学习器叠起来，做出一个强学习器**。

AdaBoost（Adaptive Boosting，自适应提升）的核心：

1. 初始给所有样本同样的权重；
2. 用当前样本权重训练一个弱学习器（常用决策树桩）；
3. 计算该学习器的分类误差率；
4. 错分样本的权重提高，正确样本的权重降低；
5. 重复步骤 2–4，得到一系列弱学习器；
6. 最终用加权投票的方式组合这些弱学习器，
   误差率小的权重大，误差率大的权重小。

### 7.4.3 AdaBoost 代码示例（以树桩为基学习器）

In [4]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. 数据
X, y = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=5,
    random_state=0
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

# 2. 基学习器：单层决策树（决策树桩）
base_estimator = DecisionTreeClassifier(max_depth=1)

# 3. AdaBoost
clf = AdaBoostClassifier(
    estimator=base_estimator,   # sklearn <1.2 用 base_estimator
    n_estimators=50,
    learning_rate=1.0,
    random_state=0
)
clf.fit(X_train, y_train)

# 4. 评估
y_pred = clf.predict(X_test)
print("AdaBoost 测试集准确率：", accuracy_score(y_test, y_pred))

AdaBoost 测试集准确率： 0.8266666666666667


Boosting（如 AdaBoost、梯度提升树、XGBoost、LightGBM）适合追求较高精度的任务，
尤其是结构化数据（表格数据）的分类回归。

### 7.4.4 随机森林（Random Forest，Bagging 代表）

随机森林是以 **决策树** 为基学习器的 Bagging 变体，有两个“随机”来源：

1. **样本层面**：对原始数据做有放回抽样（Bootstrap），给每棵树一个不同的样本子集；
2. **特征层面**：在每个节点只从随机抽取的一小部分特征中选择最优特征（而不是从所有特征中选）。

参数 $k$ 控制每次节点划分使用的特征数：

* 若 $k = d$（特征总数），就退化为普通决策树；
* 若 $k = 1$，每次只随机选一个特征划分，随机性最大；
* 常见推荐：$k \approx \log_2 d$。

这样做可以让各棵树差异更大，减小它们之间的相关性，从而提高整体集成效果。

### 7.4.5 随机森林代码示例

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. 数据
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

# 2. 随机森林
clf = RandomForestClassifier(
    n_estimators=100,    # 树的数量
    max_depth=None,     # 不限制深度，可配合 min_samples_leaf 等预剪枝
    max_features="sqrt",# 每个节点使用的特征数
    random_state=0
)
clf.fit(X_train, y_train)

# 3. 评估
y_pred = clf.predict(X_test)
print("随机森林 测试集准确率：", accuracy_score(y_test, y_pred))

随机森林 测试集准确率： 0.9814814814814815


**应用场景：**
随机森林简单稳定、对参数不太敏感，是结构化数据上非常常用的“默认强力模型”。

### 本章小结

* **朴素贝叶斯**：概率模型，假设特征独立，简单高效，常用于文本分类；
* **决策树**：规则型模型，可解释性强，会过拟合，需要剪枝；
* **SVM**：寻找最大间隔超平面，支持核函数处理非线性问题；
* **集成学习**：

  * Boosting（如 AdaBoost）：顺序学习，减偏差；
  * Bagging（如随机森林）：并行学习，减方差；
  * Stacking：再训练一个“元模型”融合多个模型。

这些算法在实际工程中非常常见，可以作为你在“线性模型”之外的几个重要备选方案。